In [ ]:
import pandas as pd
import json

# Load all CSVs
roads = pd.read_csv("../data/road_network.csv")
traffic = pd.read_csv("../data/traffic_sensor_data.csv")
weather = pd.read_csv("../data/weather_conditions.csv")
events = pd.read_csv("../data/traffic_events.csv")

# Load JSON
with open("../data/sensor_locations.json") as f:
    sensors = json.load(f)

# Convert sensors JSON into DataFrame
sensors_df = pd.DataFrame(sensors["sensors"])



In [ ]:
# Merge sensors with roads
roads_sensors = sensors_df.merge(roads, left_on="road_segment_id", right_on="road_id", how="left")
print(roads_sensors.head())

# Merge traffic with sensors
traffic_full = traffic.merge(sensors_df, on="sensor_id", how="left")
print(traffic_full.head())

# Merge traffic with weather
traffic_weather = traffic_full.merge(weather, on="timestamp", how="left")

# Check events columns before merging
print(events.columns)

# Example merge (adjust keys if needed)
traffic_all = traffic_weather.merge(
    events,
    left_on="timestamp",
    right_on="timestamp_start",
    how="left"
)

print(traffic_all.head())
print(traffic_all.info())

In [ ]:
print(traffic_all.head())
print(traffic_all.info())


In [ ]:
import pandas as pd

# Road network data
roads = pd.read_csv("../data/road_network.csv")
print("Roads:", roads.shape)
print(roads.head())

# Traffic sensor data
traffic = pd.read_csv("../data/traffic_sensor_data.csv")
print("Traffic:", traffic.shape)
print(traffic.head())

# Weather conditions
weather = pd.read_csv("../data/weather_conditions.csv")
print("Weather:", weather.shape)
print(weather.head())

# Traffic events
events = pd.read_csv("../data/traffic_events.csv")
print("Events:", events.shape)
print(events.head())


In [ ]:
import json

with open("../data/sensor_locations.json") as f:
    sensors = json.load(f)

# If it's a dictionary, show keys
print(sensors.keys())

# If you want to peek inside
for key, value in list(sensors.items())[:2]:
    print(key, value)


In [ ]:
traffic.info()


In [ ]:
weather = pd.read_csv("../data/weather_conditions.csv")
print(weather.head())
events = pd.read_csv("../data/traffic_events.csv")
print(events.head())


In [ ]:
traffic.describe()


In [ ]:
traffic.isna().sum()


In [ ]:
traffic_all["timestamp"] = pd.to_datetime(traffic_all["timestamp"])
traffic_all["hour"] = traffic_all["timestamp"].dt.hour
traffic_all["day_of_week"] = traffic_all["timestamp"].dt.dayofweek
traffic_all["is_weekend"] = traffic_all["day_of_week"].isin([5,6]).astype(int)


In [17]:
import pandas as pd
import json
import numpy as np

# -----------------------------
# 1. Load datasets
# -----------------------------
roads = pd.read_csv("../data/road_network.csv")
traffic = pd.read_csv("../data/traffic_sensor_data.csv")
weather = pd.read_csv("../data/weather_conditions.csv")
events = pd.read_csv("../data/traffic_events.csv")

with open("../data/sensor_locations.json") as f:
    sensors = json.load(f)

sensors_df = pd.DataFrame(sensors["sensors"])

# -----------------------------
# 2. Merge sensors with roads
# -----------------------------
roads_sensors = sensors_df.merge(
    roads, left_on="road_segment_id", right_on="road_id", how="left"
)

# -----------------------------
# 3. Merge traffic with sensors+roads
# -----------------------------
traffic_full = traffic.merge(roads_sensors, on="sensor_id", how="left")

# -----------------------------
# 4. Merge traffic with weather (chunked to avoid MemoryError)
# -----------------------------
traffic_full["timestamp"] = pd.to_datetime(traffic_full["timestamp"])
weather["timestamp"] = pd.to_datetime(weather["timestamp"])

chunks = []
for chunk in np.array_split(traffic_full, 10):  # split into 10 smaller parts
    merged = chunk.merge(weather, on="timestamp", how="left")
    chunks.append(merged)

traffic_weather = pd.concat(chunks, ignore_index=True)

# -----------------------------
# 5. Merge traffic with events (only essential columns)
# -----------------------------
events_small = events[["road_id", "event_type", "severity"]]
traffic_all = traffic_weather.merge(events_small, on="road_id", how="left")

# -----------------------------
# 6. Feature Engineering
# -----------------------------
traffic_all["hour"] = traffic_all["timestamp"].dt.hour
traffic_all["day_of_week"] = traffic_all["timestamp"].dt.dayofweek
traffic_all["is_weekend"] = traffic_all["day_of_week"].isin([5,6]).astype(int)

traffic_all["district"] = traffic_all["district"].astype("category").cat.codes
traffic_all["surface_condition"] = traffic_all["surface_condition"].astype("category").cat.codes
traffic_all["event_type"] = traffic_all["event_type"].astype("category").cat.codes

traffic_all["vehicle_count_lag1"] = traffic_all["vehicle_count"].shift(1)
traffic_all["vehicle_count_lag2"] = traffic_all["vehicle_count"].shift(2)

# -----------------------------
# 7. Inspect final dataset
# -----------------------------
print(traffic_all.head())
print(traffic_all.info())


c:\Users\User\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


MemoryError: Unable to allocate 7.33 MiB for an array with shape (8, 120030) and data type float64